# 08_pipeline.ipynb

## Sprint 2 - PIPE: Integración del pipeline reproducible

En este notebook se integran las transformaciones desarrolladas en las etapas previas del Sprint 2 en un flujo reproducible basado en `Pipeline` y `ColumnTransformer`.

### Objetivo
Construir y validar un pipeline reutilizable para las siguientes etapas del proyecto, asegurando que las transformaciones puedan aplicarse de forma consistente sobre train, test y datos nuevos.

### Inputs
- `data/processed/train_balanced.csv`
- `data/processed/test_original.csv`

### Outputs
- `src/preprocessing.py`
- `models/preprocessor.pkl`

## 1. Importación de librerías

Se importan las librerías necesarias para:
- cargar los datasets procesados
- construir el pipeline
- validar `fit` y `transform`
- persistir el artefacto final

In [4]:
import os
import json
import pandas as pd
import numpy as np

import os
import sys

sys.path.append(os.path.abspath(".."))
print(sys.path[-1])

from src.preprocessing import (
    FeatureBuilder,
    get_column_groups,
    build_preprocessor,
    save_artifact,
)

/Users/alexandralozano/dp261-g1


## 2. Carga de datasets procesados

Se cargan los outputs generados en PB-09:
- conjunto de entrenamiento balanceado
- conjunto de prueba con distribución original

Estos serán usados para validar que el pipeline se ajusta correctamente sobre train y se aplica sobre test sin errores.

In [5]:
train_df = pd.read_csv("../data/processed/train_balanced.csv")
test_df = pd.read_csv("../data/processed/test_original.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (102410, 60)
Test shape: (14597, 60)


## 3. Separación de variables predictoras y target

Se separa la variable objetivo del resto de columnas para preparar los conjuntos que entrarán al pipeline.

In [6]:
target = "IsBadBuy"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

## 4. Aplicación opcional de feature engineering reutilizable

En esta sección se aplica un transformador de features determinísticas para reproducir parte del trabajo realizado en PB-08 dentro del flujo reutilizable del proyecto.

In [7]:
feature_builder = FeatureBuilder()

X_train_fb = feature_builder.fit_transform(X_train)
X_test_fb = feature_builder.transform(X_test)

print("Train después de FeatureBuilder:", X_train_fb.shape)
print("Test después de FeatureBuilder:", X_test_fb.shape)

Train después de FeatureBuilder: (102410, 64)
Test después de FeatureBuilder: (14597, 64)


## 5. Identificación de columnas numéricas y categóricas

Se identifican los grupos de variables para aplicar transformaciones distintas según su tipo:
- imputación + escalado para numéricas
- imputación + one-hot encoding para categóricas

In [8]:
tmp_df = X_train_fb.copy()
tmp_df[target] = y_train.values
num_cols, cat_cols = get_column_groups(tmp_df, target)

print("Numéricas:", len(num_cols))
print("Categóricas:", len(cat_cols))

## 6. Construcción del preprocessor

Se construye un `ColumnTransformer` que integra:
- imputación y escalado para variables numéricas
- imputación y codificación one-hot para variables categóricas

Este objeto será persistido y reutilizado en los sprints posteriores.

In [9]:
preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 7. Validación de `fit` y `transform`

Se valida que el preprocessor pueda:
- ajustarse correctamente sobre el conjunto de entrenamiento
- transformar el conjunto de prueba sin errores
- generar una salida consistente y reutilizable

In [10]:
X_train_t = preprocessor.fit_transform(X_train_fb)
X_test_t = preprocessor.transform(X_test_fb)

/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['vehicle_age_group']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['vehicle_age_group']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


In [11]:
print("Shape transformado train:", X_train_t.shape)
print("Shape transformado test:", X_test_t.shape)

Shape transformado train: (102410, 646)
Shape transformado test: (14597, 646)


## 8. Revisión básica de la salida transformada

Se inspecciona la dimensionalidad de la salida para confirmar que el pipeline aplicó correctamente las transformaciones esperadas.

In [14]:
print(type(X_train_t))
print(type(X_test_t))

feature_names = preprocessor.get_feature_names_out()
print("Número de features finales:", len(feature_names))
print(feature_names[:20])

<class 'scipy.sparse._csr.csr_matrix'>
<class 'scipy.sparse._csr.csr_matrix'>
Número de features finales: 646
['num__VehYear' 'num__VehicleAge' 'num__Make' 'num__Model' 'num__Trim'
 'num__SubModel' 'num__WheelTypeID' 'num__VehOdo' 'num__Size'
 'num__MMRAcquisitionAuctionAveragePrice'
 'num__MMRAcquisitionAuctionCleanPrice'
 'num__MMRAcquisitionRetailAveragePrice'
 'num__MMRAcquisitonRetailCleanPrice' 'num__MMRCurrentAuctionAveragePrice'
 'num__MMRCurrentAuctionCleanPrice' 'num__MMRCurrentRetailAveragePrice'
 'num__MMRCurrentRetailCleanPrice' 'num__BYRNO' 'num__VNZIP1'
 'num__VehBCost']


## 9. Persistencia del pipeline

Se guarda el artefacto final con `joblib` para reutilizarlo en los sprints de modelado y despliegue.

In [15]:
os.makedirs("../models", exist_ok=True)

artifact = {
    "feature_builder": feature_builder,
    "preprocessor": preprocessor
}

save_artifact(artifact, "../models/preprocessor.pkl")

print("Artefacto guardado en ../models/preprocessor.pkl")

Artefacto guardado en ../models/preprocessor.pkl


## 10. Resumen final

En esta etapa se logró:
- integrar el preprocesamiento en un objeto reproducible
- validar `fit` y `transform` sobre train y test
- persistir el pipeline para los siguientes sprints

Con esto, el Sprint 2 queda listo para que el Sprint 3 se enfoque exclusivamente en entrenar y comparar modelos.